# TinyTTS: train the G2P fallback model (dictionary model) on Kaggle

Trains or fine-tunes the small neural grapheme-to-phoneme model behind `dictionary_model.bin`
of [TinyTTS](https://github.com/pschatzmann/TinyTTS) (a GRU encoder-decoder that pronounces
words missing from the dictionary). When training is done it rebuilds everything that depends
on the model: `dictionary_model.bin`, `cmudict_exceptions.json`, `cmudict.bin` and
`cmudict_slim.bin`. Copy them over `data/default_*.bin` in your checkout, as step 6 describes.

This is a different model from the voice model of `tinytts_ljspeech16k.ipynb`, and it is much
smaller: about 833K parameters, and a few minutes per epoch even on a CPU.

**Before running**, in the notebook's sidebar under *Settings*:
- **Internet:** *On*. Your Kaggle account must be phone-verified for this. The notebook clones
  GitHub (this repository and the upstream `tiny-tts` data files).
- **Accelerator:** optional. A GPU makes each epoch much faster, but the model is small enough for
  the CPU.

**How to run:** use *Save Version → Save & Run All (Commit)*, or *Run All* in the editor.
Everything written to `/kaggle/working` is the version's output.

**Continuing in another session:** click *Add Input*, add this notebook's own previous output
(Notebook Output Files), and run it again. It finds the earlier `g2p_last.pt` automatically
(`PREVIOUS_OUTPUT = "auto"`) and resumes where it stopped.

## 1. Settings

In [ ]:
import os

REPO_URL = "https://github.com/pschatzmann/TinyTTS"
REPO_BRANCH = "main"
# Upstream data (tronghieuit/tiny-tts: cmudict.json, g2p_model.json), same pinned commit as the Docker image.
UPSTREAM_URL = "https://github.com/tronghieuit/tiny-tts.git"
UPSTREAM_COMMIT = "77c278f7a4659181bd7bcf5c5e07c768ae765147"

EPOCHS = 60          # total epochs across all sessions
BATCH_SIZE = 256
LR = 5e-4            # peak learning rate; use 2e-3 together with SCRATCH = True
HOLDOUT = 0.0        # fraction of words held out to measure generalization; 0 = train on all words
SCRATCH = False      # True: random initial weights instead of fine-tuning the shipped model

# Continue a previous session: "auto" looks for g2p/g2p_last.pt below /kaggle/input
# (add the previous notebook version's output as input); None always starts fresh.
PREVIOUS_OUTPUT = "auto"

# Publish the result as a new version of a Kaggle Model (kaggle.com/models), see step 5.
PUBLISH_MODEL = True
MODEL_HANDLE = "pschatzmann/tinytts/other/g2p-dictionary-model"   # <your Kaggle user>/<model>/<framework>/<variation>
MODEL_LICENSE = "Apache 2.0"                                      # same license as TinyTTS

# Kaggle paths (overridable via environment variables for local testing).
WORK = os.environ.get("TINYTTS_WORK", "/kaggle/working")
INPUT = os.environ.get("TINYTTS_INPUT", "/kaggle/input")
REPO = os.environ.get("TINYTTS_REPO_DIR", "/tmp/TinyTTS")   # code lives outside /kaggle/working (not part of the output)
TRAINING = f"{REPO}/research/training"
RUN_DIR = f"{WORK}/g2p"            # g2p_model.json, checkpoints, train.log
EXPORT_DIR = f"{RUN_DIR}/export"   # the rebuilt .bin files

## 2. Accelerator check

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"  cuda:0 {p.name}, {p.total_memory / 2**30:.1f} GB")
else:
    print("no GPU: training runs on the CPU, about 4 minutes per epoch")

## 3. Code and dependencies

In [ ]:
import glob
import shutil
import subprocess
import sys


def run(cmd, cwd=None, log=None, filter_fn=None, env=None):
    """Runs a command, streaming its output (optionally filtered and/or teed to a log file)."""
    print("$", " ".join(cmd), flush=True)
    logf = open(log, "a") if log else None
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if logf:
            logf.write(line)
            logf.flush()
        if filter_fn is None or filter_fn(line):
            print(line, end="", flush=True)
    if logf:
        logf.close()
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}: {' '.join(cmd)}")


if not os.path.isdir(f"{REPO}/.git") and not os.environ.get("TINYTTS_REPO_DIR"):
    run(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO])
# This notebook needs training/train_g2p.py.
if not os.path.exists(f"{TRAINING}/train_g2p.py"):
    raise SystemExit(f"{REPO_URL} ({REPO_BRANCH}) is older than this notebook -- push "
                     "research/training/train_g2p.py first.")

upstream = f"{REPO}/research/tiny-tts"
if not os.path.exists(f"{upstream}/npm-package/g2p_model.json"):
    # Sparse + blobless: only tiny_tts/ and npm-package/, not the large checkpoints/ONNX files in that repo.
    os.makedirs(upstream, exist_ok=True)
    for cmd in (["git", "init", "-q"], ["git", "remote", "add", "origin", UPSTREAM_URL],
                ["git", "sparse-checkout", "set", "tiny_tts", "npm-package"],
                ["git", "fetch", "-q", "--depth", "1", "--filter=blob:none", "origin", UPSTREAM_COMMIT],
                ["git", "checkout", "-q", "FETCH_HEAD"]):
        run(cmd, cwd=upstream)

# Kaggle already ships torch -- install everything else, never torch. The export scripts import
# tiny_tts.text.symbols, which pulls in the same dependencies as the voice training.
reqs = [l.strip() for l in open(f"{TRAINING}/requirements.txt")
        if l.strip() and not l.startswith("#") and not l.startswith(("torch", "torchaudio"))]
run([sys.executable, "-m", "pip", "install", "-q", *reqs])
run([sys.executable, "-c", "import sys; sys.path.insert(0, 'tiny-tts'); from tiny_tts.text.symbols import symbols; print(len(symbols), 'symbols')"],
    cwd=f"{REPO}/research", filter_fn=lambda l: "Warning" not in l and "warnings.warn" not in l)
print("ok")

## 4. Train and rebuild

Runs `train_g2p.py`. It prints the loss every 50 steps and the word accuracy after every epoch.
At the end it rebuilds `dictionary_model.bin`, `cmudict_exceptions.json`, `cmudict.bin` and
`cmudict_slim.bin` into `g2p/export/` (the `cmudict_exceptions.json` step takes about 10 minutes).

In [ ]:
os.makedirs(RUN_DIR, exist_ok=True)


def find_previous():
    """g2p_last.pt from a previous session's output in /kaggle/input, or None."""
    if PREVIOUS_OUTPUT is None:
        return None
    hits = glob.glob(f"{INPUT}/**/g2p/g2p_last.pt", recursive=True) if PREVIOUS_OUTPUT == "auto" \
        else glob.glob(f"{PREVIOUS_OUTPUT}/**/g2p/g2p_last.pt", recursive=True)
    return os.path.dirname(sorted(hits)[-1]) if hits else None


resume = os.path.exists(f"{RUN_DIR}/g2p_last.pt")
if not resume and (prev := find_previous()):
    print("resuming from", prev)
    for name in ("g2p_last.pt", "g2p_best.pt"):
        if os.path.exists(f"{prev}/{name}"):
            shutil.copy(f"{prev}/{name}", RUN_DIR)
    resume = True

cmd = [sys.executable, "train_g2p.py", "--out-dir", RUN_DIR, "--epochs", str(EPOCHS),
       "--batch-size", str(BATCH_SIZE), "--lr", str(LR), "--holdout", str(HOLDOUT)]
if SCRATCH and not resume:
    cmd.append("--scratch")
if resume:
    cmd.append("--resume")
run(cmd, cwd=TRAINING, log=f"{RUN_DIR}/train.log")

## 5. Check the result and publish

In [ ]:
import datetime

sys.path.insert(0, f"{REPO}/research")
os.environ["PYTHONWARNINGS"] = "ignore"
from g2p_reference_numpy import DictionaryModel

model = DictionaryModel(f"{RUN_DIR}/g2p_model.json")
for word in ["hello", "arduino", "tinytts", "microcontroller"]:
    print(f"{word:16s}", " ".join(model.predict(word)))

files = ["dictionary_model.bin", "cmudict_slim.bin", "cmudict.bin"]
for name in files:
    print(f"{name:24s} {os.path.getsize(f'{EXPORT_DIR}/{name}') / 1024 / 1024:.2f} MB")

model_dir = f"{WORK}/model"
shutil.rmtree(model_dir, ignore_errors=True)
os.makedirs(model_dir)
for name in files:
    shutil.copy(f"{EXPORT_DIR}/{name}", model_dir)
shutil.copy(f"{RUN_DIR}/g2p_last.pt", model_dir)
readme = f"""# TinyTTS G2P fallback model

Neural grapheme-to-phoneme model and matching dictionaries for [TinyTTS]({REPO_URL})
({datetime.date.today().isoformat()}, {EPOCHS} epochs).

| File | Use |
|---|---|
| `dictionary_model.bin` | Copy to `data/default_dictionary_model.bin` (`tts.setDictionaryModel(...)`) |
| `cmudict_slim.bin` | Copy to `data/default_cmudict_slim.bin`. Contains exactly the words this model gets wrong, so it must be used together with this `dictionary_model.bin`. |
| `cmudict.bin` | Copy to `data/default_cmudict.bin` (full dictionary, independent of the model) |
| `g2p_last.pt` | Training checkpoint (PyTorch) |

Then run `research/export_headers.py` in the TinyTTS checkout. See `docs/training.md` there.
"""
with open(f"{model_dir}/README.md", "w") as f:
    f.write(readme)
print("model files:", sorted(os.listdir(model_dir)))

if PUBLISH_MODEL:
    try:
        from kaggle_secrets import UserSecretsClient  # only exists on Kaggle
        os.environ.setdefault("KAGGLE_API_TOKEN", UserSecretsClient().get_secret("KAGGLE_API_TOKEN"))
    except Exception as e:
        print("no KAGGLE_API_TOKEN notebook secret:", type(e).__name__, "-- trying without it")
    try:
        run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kagglehub"])
        import kagglehub

        kagglehub.model_upload(MODEL_HANDLE, model_dir, license_name=MODEL_LICENSE,
                               version_notes=f"{EPOCHS} epochs")
        print(f"published: https://www.kaggle.com/models/{MODEL_HANDLE.rsplit('/', 2)[0]}")
    except Exception as e:
        print(f"model upload failed ({type(e).__name__}: {e})")
        print("Add the KAGGLE_API_TOKEN secret (see below) and check MODEL_HANDLE's owner; "
              f"the files are in {model_dir} either way.")

Uploading needs your Kaggle API token as a **notebook secret**: in the editor open *Add-ons →
Secrets*, add a secret named `KAGGLE_API_TOKEN` with the token (*kaggle.com → Settings → API →
Create New Token*), and tick it for this notebook. Without it, or if the upload fails, the step
above only prints what's missing; the files are still in the output. If the Kaggle Model doesn't
exist yet, it is created as a **private** model.

## 6. Using the result

Download the three `.bin` files from the saved version's **Output** tab (`g2p/export/`), or from
the published model. Then, in your TinyTTS checkout:

```bash
cp dictionary_model.bin data/default_dictionary_model.bin
cp cmudict_slim.bin     data/default_cmudict_slim.bin
cp cmudict.bin          data/default_cmudict.bin
cd research && python export_headers.py       # refresh the C++ headers
```

Always replace `dictionary_model.bin` and `cmudict_slim.bin` together: the slim dictionary only
holds the words that this particular model gets wrong.

`g2p/train.log` is the full training log.